# Data Merge, Join, Concatenate and Compare

Pandas provides various methods for combining and comparing Series or DataFrame.

- `concat()`: Merge multiple Series or DataFrame objects along a shared index or column.

- `merge()`: Combine two Series or DataFrame objects with SQL-style joining.

- `DataFrame.join()`: Merge multiple DataFrame objects along the columns.

- `Series.compare()` and `DataFrame.compare()`: Show differences in values between two Series or DataFrame objects

In [254]:
import pandas as pd
import numpy as np
rng = np.random.default_rng(seed=100)

### Generation of DataFrames.

In [255]:
# Generation of dataframes.
df1 = pd.DataFrame({
    'A': rng.integers(-100, 200, 4),
    'B': rng.integers(-100, 200, 4),
    'C': rng.integers(-100, 200, 4),
    'D': rng.integers(-100, 200, 4),
}, index=[0, 1, 2, 3])

df1

,A,B,C,D
0,130,-76,76,23
1,150,-14,192,137
2,-63,33,184,177
3,78,-88,78,173


In [256]:
df2 = pd.DataFrame({
    'A': rng.integers(-100, 200, 4),
    'B': rng.integers(-100, 200, 4),
    'C': rng.integers(-100, 200, 4),
    'D': rng.integers(-100, 200, 4),
}, index=[4, 5, 6, 7])

df2

,A,B,C,D
4,-99,51,-4,71
5,106,194,88,79
6,115,106,-57,132
7,-44,-15,74,60


In [257]:
df3 = pd.DataFrame({
    'A': rng.integers(-100, 200, 4),
    'B': rng.integers(-100, 200, 4),
    'C': rng.integers(-100, 200, 4),
    'D': rng.integers(-100, 200, 4),
}, index=[4, 5, 6, 7])

df3

,A,B,C,D
4,117,-63,-32,164
5,198,131,199,18
6,-62,-58,29,34
7,50,48,193,-4


## Concat

The `concat()` function concatenates an arbitrary amount of Series or DataFrame objects along an axis while performing optional set logic (union or intersection) of the indexes on the other axes.

In [258]:
# concat() takes a list or dict of homogeneously-typed objects and concatenates them.
frame = [df1, df2, df3]
result = pd.concat(frame)
result

,A,B,C,D
0,130,-76,76,23
1,150,-14,192,137
2,-63,33,184,177
3,78,-88,78,173
4,-99,51,-4,71
5,106,194,88,79
6,115,106,-57,132
7,-44,-15,74,60
4,117,-63,-32,164
5,198,131,199,18


#### Joining logic of the resulting axis. 

In [259]:
# Define Dataframe.
df4 = pd.DataFrame({
    'B': rng.random((4, )),
    'D': rng.random((4, )),
    'F': rng.random((4, )),
}, index=[2, 3, 6, 7])

df4

,B,D,F
2,0.862181,0.389765,0.274573
3,0.799325,0.131648,0.656117
6,0.691431,0.625497,0.014676
7,0.408518,0.082401,0.834532


In [260]:
# axis = 0: Row-wise, Default.
# axis = 1: Column-wise.
result = pd.concat([df1, df4], axis=1)
result

,A,B,C,D,B,D,F
0,130.0,-76.0,76.0,23.0,NaN,NaN,NaN
1,150.0,-14.0,192.0,137.0,NaN,NaN,NaN
2,-63.0,33.0,184.0,177.0,0.862181,0.389765,0.274573
3,78.0,-88.0,78.0,173.0,0.799325,0.131648,0.656117
6,NaN,NaN,NaN,NaN,0.691431,0.625497,0.014676
7,NaN,NaN,NaN,NaN,0.408518,0.082401,0.834532


In [261]:
# join='inner' takes the intersection of the axis values.
result = pd.concat([df1, df4], axis=1, join="inner")
result

,A,B,C,D,B,D,F
2,-63,33,184,177,0.862181,0.389765,0.274573
3,78,-88,78,173,0.799325,0.131648,0.656117


In [262]:
# To perform an effective “left” join using the exact index from the original DataFrame, 
# result can be reindexed.
result = pd.concat([df1, df4], axis=1).reindex(df1.index)
result

,A,B,C,D,B,D,F
0,130.0,-76.0,76.0,23.0,NaN,NaN,NaN
1,150.0,-14.0,192.0,137.0,NaN,NaN,NaN
2,-63.0,33.0,184.0,177.0,0.862181,0.389765,0.274573
3,78.0,-88.0,78.0,173.0,0.799325,0.131648,0.656117


#### Ignoring indexes on the concatenation axis.

In [263]:
# ignore_index ignores overlapping indexes.
result = pd.concat([df1, df4], ignore_index=True, sort=False)
result

,A,B,C,D,F
0,130.0,-76.000000,76.0,23.000000,NaN
1,150.0,-14.000000,192.0,137.000000,NaN
2,-63.0,33.000000,184.0,177.000000,NaN
3,78.0,-88.000000,78.0,173.000000,NaN
4,NaN,0.862181,NaN,0.389765,0.274573
5,NaN,0.799325,NaN,0.131648,0.656117
6,NaN,0.691431,NaN,0.625497,0.014676
7,NaN,0.408518,NaN,0.082401,0.834532


#### Concatenating Series and DataFrame together.

In [264]:
s1 = pd.Series(["X0", "X1", "X2", "X3"], name="X")
result = pd.concat([df1, s1], axis=1)
result

,A,B,C,D,X
0,130,-76,76,23,X0
1,150,-14,192,137,X1
2,-63,33,184,177,X2
3,78,-88,78,173,X3


In [265]:
# ignore_index=True will drop all name references.
result = pd.concat([df1, s1], axis=1, ignore_index=True)
result

,0,1,2,3,4
0,130,-76,76,23,X0
1,150,-14,192,137,X1
2,-63,33,184,177,X2
3,78,-88,78,173,X3


#### Appending rows to a DataFrame

In [266]:
s2 = pd.Series(rng.integers(-300, 300, 4), index=['A', 'B', 'C', 'D'], name='X')
s2

A    -13
B   -257
C   -163
D     14
Name: X, dtype: int64

In [267]:
# to_frame() converts a Series into DataFrame.
s2.to_frame()

,X
A,-13
B,-257
C,-163
D,14


In [268]:
# Traspose the dataframe.
s2.to_frame().T

,A,B,C,D
X,-13,-257,-163,14


In [269]:
result = pd.concat([df1, s2.to_frame().T], ignore_index=True)
result

,A,B,C,D
0,130,-76,76,23
1,150,-14,192,137
2,-63,33,184,177
3,78,-88,78,173
4,-13,-257,-163,14


#### Resulting keys

The keys argument adds another axis level to the resulting index or column 

In [270]:
result = pd.concat(frame, keys=["x", "y", "z"])
result

A    B    C    D
x 0  130  -76   76   23
  1  150  -14  192  137
  2  -63   33  184  177
  3   78  -88   78  173
y 4  -99   51   -4   71
  5  106  194   88   79
  6  115  106  -57  132
  7  -44  -15   74   60
z 4  117  -63  -32  164
  5  198  131  199   18
  6  -62  -58   29   34
  7   50   48  193   -4

In [271]:
result.loc['y']

,A,B,C,D
4,-99,51,-4,71
5,106,194,88,79
6,115,106,-57,132
7,-44,-15,74,60


## merge()

The `merge()` function performs join operations similar to relational databases like SQL.

In [272]:
# Generating Data Frames.
dept_df = pd.DataFrame({
    'DeptID': [f'D{idx}' for idx in range(201, 208)],
    'Department': ['Engineering', 'Operations', 'Sales & Marketing', 'HR', 'Finanace', 'Product', 'Canteen']
})

dept_df

,DeptID,Department
0,D201,Engineering
1,D202,Operations
2,D203,Sales & Marketing
3,D204,HR
4,D205,Finanace
5,D206,Product
6,D207,Canteen


In [273]:
emp_df = pd.DataFrame({
    'EmpID': ['A' + str(i) for i in range(101, 113)],
    'EmpName': ['Abby', 'Bobby', 'Cathy', 'Duck', 'Emma', 'Fiddy', 'Gimmy', 'Harry', 'Indu', 'Jack', 'Kim', 'Lima'],
    'Salary': [3498, 3476, 6578, 3988, 7866, 10000, 6500, 7655, 8900, 2500, 2000, 3200],
    'DeptID': ['D201', 'D201', 'D201', 'D204', 'D204', 'D205', 'D203', 'D203', 'D204', 'D202', None, 'D208']
})

emp_df

,EmpID,EmpName,Salary,DeptID
0,A101,Abby,3498,D201
1,A102,Bobby,3476,D201
2,A103,Cathy,6578,D201
3,A104,Duck,3988,D204
4,A105,Emma,7866,D204
5,A106,Fiddy,10000,D205
6,A107,Gimmy,6500,D203
7,A108,Harry,7655,D203
8,A109,Indu,8900,D204
9,A110,Jack,2500,D202


In [274]:
# By default, it shows inner join. how = 'inner'
result = pd.merge(emp_df, dept_df, on='DeptID')
result

,EmpID,EmpName,Salary,DeptID,Department
0,A101,Abby,3498,D201,Engineering
1,A102,Bobby,3476,D201,Engineering
2,A103,Cathy,6578,D201,Engineering
3,A104,Duck,3988,D204,HR
4,A105,Emma,7866,D204,HR
5,A106,Fiddy,10000,D205,Finanace
6,A107,Gimmy,6500,D203,Sales & Marketing
7,A108,Harry,7655,D203,Sales & Marketing
8,A109,Indu,8900,D204,HR
9,A110,Jack,2500,D202,Operations


In [275]:
# Left ourter join. how = 'left'
result = pd.merge(emp_df, dept_df, on='DeptID', how='left')
result

,EmpID,EmpName,Salary,DeptID,Department
0,A101,Abby,3498,D201,Engineering
1,A102,Bobby,3476,D201,Engineering
2,A103,Cathy,6578,D201,Engineering
3,A104,Duck,3988,D204,HR
4,A105,Emma,7866,D204,HR
5,A106,Fiddy,10000,D205,Finanace
6,A107,Gimmy,6500,D203,Sales & Marketing
7,A108,Harry,7655,D203,Sales & Marketing
8,A109,Indu,8900,D204,HR
9,A110,Jack,2500,D202,Operations


In [276]:
# right ourter join. how = 'right'
result = pd.merge(emp_df, dept_df, on='DeptID', how='right')
result

,EmpID,EmpName,Salary,DeptID,Department
0,A101,Abby,3498.0,D201,Engineering
1,A102,Bobby,3476.0,D201,Engineering
2,A103,Cathy,6578.0,D201,Engineering
3,A110,Jack,2500.0,D202,Operations
4,A107,Gimmy,6500.0,D203,Sales & Marketing
5,A108,Harry,7655.0,D203,Sales & Marketing
6,A104,Duck,3988.0,D204,HR
7,A105,Emma,7866.0,D204,HR
8,A109,Indu,8900.0,D204,HR
9,A106,Fiddy,10000.0,D205,Finanace


In [277]:
# full ourter join. how = 'outer'
result = pd.merge(emp_df, dept_df, on='DeptID', how='outer')
result

,EmpID,EmpName,Salary,DeptID,Department
0,A101,Abby,3498.0,D201,Engineering
1,A102,Bobby,3476.0,D201,Engineering
2,A103,Cathy,6578.0,D201,Engineering
3,A110,Jack,2500.0,D202,Operations
4,A107,Gimmy,6500.0,D203,Sales & Marketing
5,A108,Harry,7655.0,D203,Sales & Marketing
6,A104,Duck,3988.0,D204,HR
7,A105,Emma,7866.0,D204,HR
8,A109,Indu,8900.0,D204,HR
9,A106,Fiddy,10000.0,D205,Finanace


In [278]:
# Cross Join: Cartesian products of rows of both frames.

left = pd.DataFrame(
   {
      "key1": ["K0", "K0", "K1", "K2"],
      "key2": ["K0", "K1", "K0", "K1"],
      "A": ["A0", "A1", "A2", "A3"],
      "B": ["B0", "B1", "B2", "B3"],
   }
)

right = pd.DataFrame(
   {
      "key1": ["K0", "K1", "K1", "K2"],
      "key2": ["K0", "K0", "K0", "K0"],
      "C": ["C0", "C1", "C2", "C3"],
      "D": ["D0", "D1", "D2", "D3"],
   }
)

result = pd.merge(left, right, how="cross")
result

,key1_x,key2_x,A,B,key1_y,key2_y,C,D
0,K0,K0,A0,B0,K0,K0,C0,D0
1,K0,K0,A0,B0,K1,K0,C1,D1
2,K0,K0,A0,B0,K1,K0,C2,D2
3,K0,K0,A0,B0,K2,K0,C3,D3
4,K0,K1,A1,B1,K0,K0,C0,D0
5,K0,K1,A1,B1,K1,K0,C1,D1
6,K0,K1,A1,B1,K1,K0,C2,D2
7,K0,K1,A1,B1,K2,K0,C3,D3
8,K1,K0,A2,B2,K0,K0,C0,D0
9,K1,K0,A2,B2,K1,K0,C1,D1


In [279]:
# Inner join on multiple keys.
result = pd.merge(left, right, how='inner', on=['key1', 'key2'])
result

,key1,key2,A,B,C,D
0,K0,K0,A0,B0,C0,D0
1,K1,K0,A2,B2,C1,D1
2,K1,K0,A2,B2,C2,D2


## DataFrame.join()

`DataFrame.join()` combines the columns of multiple, potentially differently-indexed DataFrame into a single result DataFrame.

#### Example:

**Task 1**

- Create a series `Eng_Marks` for the 4 students as follows **85, 76, 64, 87**. Label each students as **s1, s2, s3 and s4** respectively.

- Create a series `His_Marks` for the 4 students as follows **67, 77, 44, 78**. Label each students as **s1, s2, s3 and s4** respectively.

- Create a dataframe `marks_A` for the 4 students using `Eng_Marks` and `His_Marks`. The columns name should be **'English_Marks' and 'History_Marks'** respectively.

- Display `marks_A`.

**Task 2**

- Create a series `Math_Marks` for the 4 students from a 1-D array of 4 elements derived from the range between **65 to 92**. Label each students as **s2, s3, s5 and s6** respectively.

- Create a series `Sci_Marks` for the 4 students from a 1-D array of 4 elements derived from the range between **72 to 89**. Label each students as **s2, s3, s5 and s6** respectively.

- Create a dataframe `marks_B` for the 4 students using `Math_Marks` and `Sci_Marks`. The columns name should be **'Math_Marks' and 'Science_Marks'** respectively.

- Display `marks_B`.

**Task 3**

- Join the two dataframes `marks_A` and `marks_B` into `student_marks_join`, and display the result. 

In [280]:
# Task 1.
students_list1 = ['s1', 's2', 's3', 's4']
Eng_Marks = pd.Series([85, 76, 64, 87], index=students_list1)
His_Marks = pd.Series([67, 77, 44, 78], index=students_list1)
marks_A = pd.DataFrame({
    'English_Marks': Eng_Marks,
    'History_Marks': His_Marks
}, index=students_list1)

marks_A

,English_Marks,History_Marks
s1,85,67
s2,76,77
s3,64,44
s4,87,78


In [281]:
# Task 2.
students_list2 = ['s2', 's3', 's5', 's6']
Math_Marks = pd.Series(np.linspace(65, 92, num=4, dtype=np.int16), index=students_list2)
Sci_Marks = pd.Series(np.linspace(72, 89, num=4, dtype=np.int16), index=students_list2)
marks_B = pd.DataFrame({
    'Math_Marks': Math_Marks,
    'Science_Marks': Sci_Marks
}, index=students_list2)

marks_B

,Math_Marks,Science_Marks
s2,65,72
s3,74,77
s5,83,83
s6,92,89


In [282]:
# Task 3. 
student_marks_join = marks_A.join(marks_B) # Left Outer
student_marks_join

,English_Marks,History_Marks,Math_Marks,Science_Marks
s1,85,67,NaN,NaN
s2,76,77,65.0,72.0
s3,64,44,74.0,77.0
s4,87,78,NaN,NaN


In [283]:
student_marks_join = marks_A.join(marks_B, how='right') # Right Outer
student_marks_join

,English_Marks,History_Marks,Math_Marks,Science_Marks
s2,76.0,77.0,65,72
s3,64.0,44.0,74,77
s5,NaN,NaN,83,83
s6,NaN,NaN,92,89


In [284]:
student_marks_join = marks_A.join(marks_B, how='outer') # Full Outer
student_marks_join

,English_Marks,History_Marks,Math_Marks,Science_Marks
s1,85.0,67.0,NaN,NaN
s2,76.0,77.0,65.0,72.0
s3,64.0,44.0,74.0,77.0
s4,87.0,78.0,NaN,NaN
s5,NaN,NaN,83.0,83.0
s6,NaN,NaN,92.0,89.0


In [285]:
student_marks_join = marks_A.join(marks_B, how='inner') # Inner
student_marks_join

,English_Marks,History_Marks,Math_Marks,Science_Marks
s2,76,77,65,72
s3,64,44,74,77


#### Joining a single Index to a MultiIndex.

In [288]:
left_df = pd.DataFrame({
    'A': rng.integers(10, 30 ,3),
    'B': rng.integers(20, 50, 3)
}, index = pd.Index(['K0', 'K1', 'K2'], name = 'Idx'))

left_df

,A,B
Idx,,
K0,23,28
K1,21,22
K2,17,24


In [289]:
# MultiIndex.
mul_idx = pd.MultiIndex.from_tuples([("K0", "Y0"), ("K1", "Y1"), ("K2", "Y2"), ("K2", "Y3")], names = ['Idx', 'Y']) 
mul_idx

MultiIndex([('K0', 'Y0'),
            ('K1', 'Y1'),
            ('K2', 'Y2'),
            ('K2', 'Y3')],
           names=['Idx', 'Y'])

In [290]:
right_df = pd.DataFrame({
    'C': rng.integers(10, 30 ,4),
    'D': rng.integers(20, 60 ,4),
}, index=mul_idx)
right_df

C   D
Idx Y         
K0  Y0  29  43
K1  Y1  29  33
K2  Y2  21  43
    Y3  10  42

In [292]:
result = left_df.join(right_df, how="inner")
result

A   B   C   D
Idx Y                 
K0  Y0  23  28  29  43
K1  Y1  21  22  29  33
K2  Y2  17  24  21  43
    Y3  17  24  10  42

#### Joining with two MultiIndex

In [293]:
leftindex = pd.MultiIndex.from_product(
    [list("abc"), list("xy"), [1, 2]], names=["abc", "xy", "num"]
)
leftindex

MultiIndex([('a', 'x', 1),
            ('a', 'x', 2),
            ('a', 'y', 1),
            ('a', 'y', 2),
            ('b', 'x', 1),
            ('b', 'x', 2),
            ('b', 'y', 1),
            ('b', 'y', 2),
            ('c', 'x', 1),
            ('c', 'x', 2),
            ('c', 'y', 1),
            ('c', 'y', 2)],
           names=['abc', 'xy', 'num'])

In [294]:
left = pd.DataFrame({"v1": range(12)}, index=leftindex)
left

v1
abc xy num    
a   x  1     0
       2     1
    y  1     2
       2     3
b   x  1     4
       2     5
    y  1     6
       2     7
c   x  1     8
       2     9
    y  1    10
       2    11

In [295]:
rightindex = pd.MultiIndex.from_product(
    [list("abc"), list("xy")], names=["abc", "xy"]
)
rightindex

MultiIndex([('a', 'x'),
            ('a', 'y'),
            ('b', 'x'),
            ('b', 'y'),
            ('c', 'x'),
            ('c', 'y')],
           names=['abc', 'xy'])

In [296]:
right = pd.DataFrame({"v2": range(101, 107)}, index=rightindex)
right

v2
abc xy     
a   x   101
    y   102
b   x   103
    y   104
c   x   105
    y   106

In [297]:
left.join(right, on = ['abc', 'xy'])

v1   v2
abc xy num         
a   x  1     0  101
       2     1  101
    y  1     2  102
       2     3  102
b   x  1     4  103
       2     5  103
    y  1     6  104
       2     7  104
c   x  1     8  105
       2     9  105
    y  1    10  106
       2    11  106

## compare()

The `Series.compare()` and `DataFrame.compare()` methods allow you to compare two DataFrame or Series, respectively, and summarize their differences.

In [298]:
df = pd.DataFrame(
    {
        "col1": ["a", "a", "b", "b", "a"],
        "col2": [1.0, 2.0, 3.0, np.nan, 5.0],
        "col3": [1.0, 2.0, 3.0, 4.0, 5.0],
    },
    columns=["col1", "col2", "col3"],
)

df

,col1,col2,col3
0,a,1.0,1.0
1,a,2.0,2.0
2,b,3.0,3.0
3,b,NaN,4.0
4,a,5.0,5.0


In [300]:
df1 = df.copy()
df1.loc[0, 'col1'] = 'c'
df1.loc[2, "col3"] = 4.0
df1

,col1,col2,col3
0,c,1.0,1.0
1,a,2.0,2.0
2,b,3.0,4.0
3,b,NaN,4.0
4,a,5.0,5.0


In [302]:
df.compare(df1)

col1       col3      
  self other self other
0    a     c  NaN   NaN
2  NaN   NaN  3.0   4.0

In [303]:
df.compare(df1, keep_shape=True)

col1       col2       col3      
  self other self other self other
0    a     c  NaN   NaN  NaN   NaN
1  NaN   NaN  NaN   NaN  NaN   NaN
2  NaN   NaN  NaN   NaN  3.0   4.0
3  NaN   NaN  NaN   NaN  NaN   NaN
4  NaN   NaN  NaN   NaN  NaN   NaN

In [304]:
df.compare(df1, keep_shape=True, keep_equal=True)

col1       col2       col3      
  self other self other self other
0    a     c  1.0   1.0  1.0   1.0
1    a     a  2.0   2.0  2.0   2.0
2    b     b  3.0   3.0  3.0   4.0
3    b     b  NaN   NaN  4.0   4.0
4    a     a  5.0   5.0  5.0   5.0

In [307]:
df.compare(df1, align_axis=0)

col1  col3
0 self     a   NaN
  other    c   NaN
2 self   NaN   3.0
  other  NaN   4.0